# PAPILLON Structured v1 Tutorial

This notebook shows how to use the new `structured_v1` pipeline we added to the repository.

You will learn how to:
- initialize the local and cloud models,
- build the new route-aware PAPILLON pipeline,
- preview whether a query takes the `direct` or `protected` route,
- manually edit the protected cloud prompt before sending it,
- evaluate a few examples with the new deterministic metrics.


## What changed from the old notebook?

The original `papillon_tutorial.ipynb` defines the legacy PAPILLON class inline.

This notebook does **not** redefine PAPILLON in the notebook. Instead, it imports the actual repository modules:
- `pipeline_factory.py` to choose `legacy` vs `structured_v1`
- `structured_pipeline.py` for the new structured delegation flow
- `privacy_filter.py` for route selection and redaction metadata
- `run_dspy_optimization_llama.py` metrics helpers for evaluation

That means this notebook stays aligned with the current codebase.


## 1. Install dependencies

Run this once in a fresh environment. After installation, restart the kernel before continuing.


In [5]:
!pip install dspy-ai presidio-analyzer spacy pandas

Defaulting to user installation because normal site-packages is not writeable
  Using cached dspy_ai-3.1.3-py3-none-any.whl.metadata (285 bytes)
  Using cached presidio_analyzer-2.2.362-py3-none-any.whl.metadata (6.2 kB)
  Using cached spacy-3.8.14-cp313-cp313-win_amd64.whl.metadata (28 kB)
  Using cached dspy-3.1.3-py3-none-any.whl.metadata (8.4 kB)
  Using cached phonenumbers-9.0.27-py2.py3-none-any.whl.metadata (10 kB)
  Using cached tldextract-5.3.1-py3-none-any.whl.metadata (7.3 kB)
  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached murmurhash-1.0.15-cp313-cp313-win_amd64.whl.metadata (2.3 kB)
  Using cached cymem-2.0.13-cp313-cp313-win_amd64.whl.metadata (9.9 kB)
  Using cached preshed-3.0.13-cp313-cp313-win_amd64.whl.metadata (5.4 kB)
  Using cached thinc-8.3.13-cp313-cp313-win_amd64.whl.metadata (15 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using 


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
!python -m spacy download en_core_web_sm


Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     --------- ------------------------------ 2.9/12.8 MB 20.8 MB/s eta 0:00:01
     -------------------------------- ------ 10.7/12.8 MB 30.8 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 29.6 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 2. Start your trusted local model server

Before running the next cells, start your local model server in a terminal. Example:

```bash
python -m sglang.launch_server --model-path meta-llama/Llama-3.1-8B-Instruct --port 3012
```

This notebook assumes the local model is available at `http://127.0.0.1:3012/v1`.


## 3. Configure paths and runtime settings

Run this notebook from the repository root so that `papillon/` and `pupa/` are visible.


In [7]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
assert (PROJECT_ROOT / "papillon").exists(), "Run this notebook from the PAPILLON repo root."

PAPILLON_SRC = PROJECT_ROOT / "papillon"
if str(PAPILLON_SRC) not in sys.path:
    sys.path.insert(0, str(PAPILLON_SRC))

MODEL_NAME = "llama3.2:1b"
PORT_NUMBER = 11434
OPENAI_MODEL = "openrouter/openai/gpt-4o-mini"
ALLOW_DIRECT_BYPASS = True
PII_SCORE_THRESHOLD = 0.5

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "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"

print("Project root:", PROJECT_ROOT)
print("Using local model:", MODEL_NAME)
print("Using cloud model:", OPENAI_MODEL)


Project root: c:\Users\user\Desktop\AE\PAPILLON
Using local model: llama3.2:1b
Using cloud model: openrouter/openai/gpt-4o-mini


## 4. Import the current repo modules

These imports come from the current codebase, not from inline notebook definitions.


In [8]:
import dspy
import pandas as pd
from dspy import Example

from pipeline_factory import build_pipeline
from prompt_paths import parse_model_prompt
from run_dspy_optimization_llama import metric_finegrained


## 5. Initialize the local and cloud LMs

`local_lm` is your trusted hosted model.

`openai_lm` is the stronger cloud model used as the untrusted remote model.


In [9]:
local_lm = dspy.LM(
    f"openai/{MODEL_NAME}",
    api_base=f"http://127.0.0.1:{PORT_NUMBER}/v1",
    api_key="",
    max_tokens=4000,
)
dspy.configure(lm=local_lm)

openai_lm = dspy.LM(
    f"openai/{OPENAI_MODEL}",
    max_tokens=4000,
)
print("Models are initialized.")

Models are initialized.


## 6. Build the new `structured_v1` pipeline

Important behavior:
- `direct` route: if the privacy filter is available and finds no PII with enough confidence, the original query is sent directly to the cloud model.
- `protected` route: if PII is detected, or detection is unavailable/uncertain, the local model generates a fixed-schema prompt with `Task`, `Context`, and `Style`, then the cloud model answers that safer prompt, and the local model synthesizes the final answer.


In [10]:
structured_papillon = build_pipeline(
    pipeline_name="structured_v1",
    untrusted_model=openai_lm,
    allow_direct_bypass=ALLOW_DIRECT_BYPASS,
    privacy_filter_name="regex_presidio",
    pii_score_threshold=PII_SCORE_THRESHOLD,
)

print("structured_v1 pipeline is ready.")


structured_v1 pipeline is ready.


## 7. Preview route selection before inference

The `preview()` method is one of the biggest differences from the old notebook.

It lets you inspect:
- whether the route is `direct` or `protected`,
- why that route was chosen,
- what PII was detected,
- the exact cloud prompt that would be sent.


In [11]:
def show_preview(query):
    preview = structured_papillon.preview(query)
    print("Query:", query)
    print("Route:", preview["route"])
    print("Reason:", preview["route_reason"])
    print("Detector available:", preview["detector_available"])
    print("Detector uncertain:", preview["detector_uncertain"])
    print("Detected PII:")
    for entity in preview["detected_pii"]:
        print("  -", entity)
    print("\nCloud prompt:\n")
    print(preview["cloud_prompt"])
    return preview

direct_query = "Summarize the attached article in three bullet points."
protected_query = "Write a professional application email to OpenAI. My name is Alice Kim, and I studied at KAIST in Seoul."


In [12]:
direct_preview = show_preview(direct_query)


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0413 20:18:34.633000 9192 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Query: Summarize the attached article in three bullet points.
Route: direct
Reason: no_pii_detected
Detector available: True
Detector uncertain: False
Detected PII:

Cloud prompt:

Summarize the attached article in three bullet points.


In [13]:
protected_preview = show_preview(protected_query)


Query: Write a professional application email to OpenAI. My name is Alice Kim, and I studied at KAIST in Seoul.
Route: protected
Reason: pii_detected
Detector available: True
Detector uncertain: False
Detected PII:
  - {'entity_type': 'LOCATION', 'text': 'OpenAI', 'start': 42, 'end': 48, 'score': 0.85, 'source': 'presidio'}
  - {'entity_type': 'PERSON', 'text': 'Alice Kim', 'start': 61, 'end': 70, 'score': 0.85, 'source': 'presidio'}
  - {'entity_type': 'ORGANIZATION', 'text': 'KAIST', 'start': 89, 'end': 94, 'score': 0.85, 'source': 'presidio'}
  - {'entity_type': 'LOCATION', 'text': 'Seoul', 'start': 98, 'end': 103, 'score': 0.85, 'source': 'presidio'}

Cloud prompt:

Task:
Request application email

Context:
{"Personalized communication as per [PERSON_1]'s study program at [ORGANIZATION_1], conducted via [LOCATION_2]." This establishes context without revealing sensitive information.}

Style:
{"Professional tone, concise language, and format ensuring clarity without divulging person

## 8. Run the pipeline normally

Calling the pipeline directly with `structured_papillon(query)` uses the automatic route decision.

This returns metadata-rich predictions, including:
- `pred.route`
- `pred.cloud_prompt`
- `pred.structured_fields`
- `pred.detected_pii`
- `pred.output`


In [14]:
direct_pred = structured_papillon(direct_query)
print("Route:", direct_pred.route)
print("Cloud prompt:\n", direct_pred.cloud_prompt)
print("Output:\n", direct_pred.output)


Route: protected
Cloud prompt:
 
Output:
 


In [15]:
protected_pred = structured_papillon(protected_query)
print("Route:", protected_pred.route)
print("Cloud prompt:\n", protected_pred.cloud_prompt)
print("Output:\n", protected_pred.output)


Route: protected
Cloud prompt:
 
Output:
 


## 9. Manually edit the protected cloud prompt

This is the notebook equivalent of the UI flow.

Use `run_with_prompt()` when you want to review or modify the protected prompt before sending it to the cloud model.

You should only do this on the `protected` route. For a `direct` route, the original query is already the cloud prompt.


In [17]:
edited_prompt = protected_preview["cloud_prompt"].replace(
    "Professional, concise, and persuasive.",
    "Professional, concise, and warm."
)

manual_pred = structured_papillon.run_with_prompt(protected_query, edited_prompt)
print("Edited cloud prompt:\n", manual_pred.cloud_prompt)
print("Output:\n", manual_pred.output)


AuthenticationError: litellm.AuthenticationError: AuthenticationError: OpenAIException - Incorrect API key provided: sk-or-v1*************************************************************e0bc. You can find your API key at https://platform.openai.com/account/api-keys.

## 10. Evaluate a few real PUPA rows

This section uses the same repo metric helper that the scripts use.

The returned metrics now include:
- `quality`
- `leakage`
- `exposed_token_count`
- `entity_retention_rate`
- `schema_valid`
- `route`


In [ ]:
def evaluate_rows(df, pipeline, limit=3):
    rows = []
    for _, row in df.head(limit).iterrows():
        gold = Example(
            {
                "target_response": row["target_response"],
                "user_query": row["user_query"],
                "pii_str": row["pii_units"],
            }
        ).with_inputs("user_query")
        pred = pipeline(row["user_query"])
        metrics = metric_finegrained(gold, pred, openai_lm)
        rows.append(
            {
                "route": metrics["route"],
                "quality": metrics["quality"],
                "leakage": metrics["leakage"],
                "exposed_token_count": metrics["exposed_token_count"],
                "entity_retention_rate": metrics["entity_retention_rate"],
                "schema_valid": metrics["schema_valid"],
                "cloud_prompt": getattr(pred, "cloud_prompt", getattr(pred, "prompt", "")),
            }
        )
    return pd.DataFrame(rows)

sample_df = pd.read_csv(PROJECT_ROOT / "pupa" / "PUPA_TNB.csv")
evaluate_rows(sample_df, structured_papillon, limit=3)


## 11. Optional: compare `legacy` vs `structured_v1`

This cell loads the old optimized legacy prompt file and compares it against the new structured flow.

Important: legacy optimized prompt JSON files should only be loaded into the `legacy` pipeline, not into `structured_v1`.


In [ ]:
legacy_papillon = build_pipeline(
    pipeline_name="legacy",
    untrusted_model=openai_lm,
)
legacy_prompt_file = parse_model_prompt(MODEL_NAME)
legacy_papillon.load(str(PAPILLON_SRC / legacy_prompt_file), use_legacy_loading=True)

legacy_pred = legacy_papillon(protected_query)

print("LEGACY prompt:\n")
print(legacy_pred.prompt)
print("\nSTRUCTURED v1 prompt:\n")
print(protected_pred.cloud_prompt)


## 12. What to do next

Once this notebook is working, you can use the repo scripts for larger runs:

Interactive use:
```bash
python papillon/run_papillon_interactive.py --port 3012 --model_name meta-llama/Llama-3.1-8B-Instruct --pipeline structured_v1
```

Batch evaluation:
```bash
python papillon/evaluate_papillon.py --port 3012 --model_name meta-llama/Llama-3.1-8B-Instruct --data_file pupa/PUPA_TNB.csv --pipeline structured_v1 --output_file_name structured_eval.csv
```

Optimization:
```bash
python papillon/run_dspy_optimization_llama.py --port 3012 --data_file pupa/PUPA_New.csv --prompt_output papillon/optimized_prompts_structured/structured_v1_demo.json --pipeline structured_v1
```

If you optimize `structured_v1`, save those artifacts to a new path such as `papillon/optimized_prompts_structured/` so they do not get mixed with the legacy prompt files.
